# Transformer Training for Computer Science Paper Abstract Generation

This notebook fine-tunes a single transformer summarization model on Computer Science papers from arXiv. The pipeline is intentionally simple:

1. collect recent CS papers from arXiv
2. extract paper body text and original abstract
3. tokenize with one transformer tokenizer
4. fine-tune one seq2seq transformer model
5. save the model for inference


In [ ]:
!pip install -q transformers datasets accelerate evaluate sentencepiece scikit-learn beautifulsoup4 lxml rouge_score

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from datasets import Dataset
import evaluate
import numpy as np
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

ARXIV_CATEGORIES = ['cs.AI', 'cs.CL', 'cs.LG', 'cs.CV']
MODEL_NAME = 'sshleifer/distilbart-cnn-12-6'
MAX_SOURCE_TOKENS = 1024
MAX_TARGET_TOKENS = 160
TRAIN_SAMPLE_LIMIT = 40
VAL_SAMPLE_LIMIT = 10
MAX_PER_CATEGORY = 15
OUTPUT_DIR = '/content/cs_transformer_summarizer'

print('Categories:', ARXIV_CATEGORIES)
print('Base model:', MODEL_NAME)

In [ ]:
def fetch_recent_paper_links(categories, max_per_category=10):
    links = []
    for category in categories:
        url = f'https://arxiv.org/list/{category}/recent?skip=0&show={max_per_category}'
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        seen = 0
        for tag in soup.find_all('a', title='View HTML'):
            href = tag.get('href')
            if href:
                links.append(href)
                seen += 1
            if seen >= max_per_category:
                break
    return links

def download_arxiv_html(arxiv_html_url):
    response = requests.get(arxiv_html_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for script in soup(['script', 'style']):
        script.extract()
    return soup

def extract_abstract_and_body(soup):
    abstract_div = soup.find('div', class_='ltx_abstract')
    abstract = abstract_div.get_text(' ', strip=True) if abstract_div else ''
    if abstract_div:
        abstract_div.extract()
    body_paragraphs = []
    for p in soup.find_all('p', class_='ltx_p'):
        text = p.get_text(' ', strip=True)
        text = re.sub(r'\[\s*(\d+\s*,?\s*)+\]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            body_paragraphs.append(text)
    body_text = ' '.join(body_paragraphs)
    abstract = abstract.replace('Abstract:', '').strip()
    return abstract, body_text

def clean_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_records(sample_limit):
    links = fetch_recent_paper_links(ARXIV_CATEGORIES, max_per_category=MAX_PER_CATEGORY)
    records = []
    for link in links:
        if len(records) >= sample_limit:
            break
        try:
            soup = download_arxiv_html(link)
            abstract, body_text = extract_abstract_and_body(soup)
            body_text = clean_text(body_text)
            if abstract and body_text:
                records.append({
                    'link': link,
                    'source_text': body_text,
                    'target_summary': abstract,
                })
        except Exception as exc:
            print(f'Skipping {link}: {exc}')
    return records

records = build_records(TRAIN_SAMPLE_LIMIT + VAL_SAMPLE_LIMIT)
print('Collected records:', len(records))
records[:1]

In [ ]:
train_records = records[:TRAIN_SAMPLE_LIMIT]
val_records = records[TRAIN_SAMPLE_LIMIT:TRAIN_SAMPLE_LIMIT + VAL_SAMPLE_LIMIT]

train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch['source_text'],
        max_length=MAX_SOURCE_TOKENS,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target_summary'],
        max_length=MAX_TARGET_TOKENS,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_dataset.map(preprocess_batch, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_batch, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    scores = rouge.compute(predictions=decoded_predictions, references=decoded_labels, use_stemmer=True)
    return {
        'rouge1': round(scores['rouge1'], 4),
        'rouge2': round(scores['rouge2'], 4),
        'rougeL': round(scores['rougeL'], 4),
    }

print(tokenized_train)
print(tokenized_val)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    predict_with_generate=True,
    logging_steps=5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    fp16=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
metrics = trainer.evaluate()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('Training complete')
print(metrics)

In [ ]:
def generate_summary(text):
    inputs = tokenizer(text, return_tensors='pt', max_length=MAX_SOURCE_TOKENS, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_length=MAX_TARGET_TOKENS,
        min_length=60,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

for sample in val_records[:3]:
    generated = generate_summary(sample['source_text'])
    print('=' * 80)
    print('Paper:', sample['link'])
    print('-' * 80)
    print('Generated Abstract:')
    print(generated)
    print('-' * 80)
    print('Reference Abstract:')
    print(sample['target_summary'])
